In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Mejos"
STUDENT_ID = "TUPM-26-3605"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)

np.set_printoptions(precision=4, suppress=True)

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

raw_measurements = np.genfromtxt(DATA_DIR / "secom.data", dtype=float)
labels = np.loadtxt(DATA_DIR / "secom_labels.data", usecols=0, dtype=int)

# Missing, constant, and usable features (same method as Task 9)
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)
all_missing = observed == 0
observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))

n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]
retained_features = [feature_names[i] for i in usable_indices]
raw_usable = raw_measurements[:, usable_indices].copy()

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT, "| ID Sum:", ID_SUM, "| Random Seed:", RANDOM_SEED)
print("Usable features:", len(retained_features))

Student: Mejos TUPM-26-3605
Seed Digit: 5 | ID Sum: 22 | Random Seed: 263605
Usable features: 474


In [2]:
n_select = 5 + (SEED_DIGIT % 3)

rng = np.random.default_rng(RANDOM_SEED)
positions = rng.choice(len(retained_features), size=n_select, replace=False)
selected = [retained_features[i] for i in positions]
X = raw_usable[:, positions]

# Feature-wise statistics (NumPy, ignoring missing values)
means = np.nanmean(X, axis=0)
medians = np.nanmedian(X, axis=0)
sds = np.nanstd(X, axis=0, ddof=1)
missing_pct = 100 * np.isnan(X).sum(axis=0) / len(X)
observed_count = (~np.isnan(X)).sum(axis=0)

# Robust standardization using median and MAD (zero MAD -> value 0, never flagged)
mad = np.nanmedian(np.abs(X - medians), axis=0)
scale = np.where(mad > 0, mad, np.inf)
robust_z = 0.6745 * (X - medians) / scale

# Flag measurements with |robust z| > 3.5
flags = np.abs(robust_z) > 3.5
flag_counts = flags.sum(axis=0)
flag_pct = 100 * flag_counts / observed_count

quality = pd.DataFrame({
    "Feature": selected,
    "Mean": means,
    "Median": medians,
    "Sample_SD": sds,
    "Missing_Pct": missing_pct,
    "MAD": mad,
    "Flagged_Count": flag_counts,
    "Flagged_Pct": flag_pct
}).sort_values(by="Flagged_Pct", ascending=False)
quality.insert(0, "Rank", list(range(1, n_select + 1)))

quality.to_csv(OUTPUT_DIR / "measurement_quality.csv", index=False)

# Verify the vectorized results against Pandas
check = pd.DataFrame(X, columns=selected)
verified = (
    np.allclose(means, check.mean().to_numpy())
    and np.allclose(medians, check.median().to_numpy())
    and np.allclose(sds, check.std().to_numpy())
    and np.allclose(missing_pct, 100 * check.isna().mean().to_numpy())
)

print("Features Selected:", n_select)
print("Selected Features:", selected)
print(quality.round(4))
print("Highest Missing Percentage:", f"{missing_pct.max():.2f}%")
print("Feature with Highest Flagged Percentage:", quality.iloc[0]["Feature"])
print("Total Flagged Observations:", flags.sum())
print("Vectorized Results Verified:", verified)

Features Selected: 7
Selected Features: ['Feature_283', 'Feature_543', 'Feature_349', 'Feature_025', 'Feature_275', 'Feature_336', 'Feature_062']
   Rank      Feature      Mean    Median  Sample_SD  Missing_Pct     MAD  \
1     1  Feature_543    0.0085    0.0078     0.0015       0.1276  0.0001   
3     2  Feature_025    1.2038    1.2830     0.1776       0.1276  0.0257   
4     3  Feature_275    8.6932    0.0784   168.9494       0.8934  0.0408   
2     4  Feature_349    0.0253    0.0240     0.0106       1.5316  0.0045   
5     5  Feature_336    9.7364    9.0736     7.5561       0.4467  0.8742   
6     6  Feature_062  116.5023  116.2118     8.6290       0.3829  4.4382   
0     7  Feature_283    2.8040    2.6580     5.8643       0.1276  0.4762   

   Flagged_Count  Flagged_Pct  
1            628      40.1278  
3            398      25.4313  
4             53       3.4127  
2             41       2.6572  
5             30       1.9231  
6             16       1.0250  
0              2     